### Chuyển video thành wav 1 kênh 16000 Hz

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from scripts.video_converter import convert_video_to_ready_wav_pydub


video_path = "../data/videos/Bai3_CongThucLuongGiac/RECORD_CongThucLuongGiac.mp4"
output_wav_path = "../data/audio/test/CongThucLuongGiac.wav"

convert_video_to_ready_wav_pydub(video_path, output_wav_path)

Đang đọc file video, vui lòng đợi...
✅ Đã chuyển đổi thành công: ../data/audio/test/CongThucLuongGiac.wav


'../data/audio/test/CongThucLuongGiac.wav'

### Chuyển audio đã convert thành văn bản transcribe

In [2]:
import math
import time
import os
from pydub import AudioSegment
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


def transcribe_long_audio(input_wav_path: str, chunk_minutes: int = 10, model: str = "qwen/qwen3-asr-1.7b") -> str:
    """
    Cat audio thanh cac doan nho, gui len API de nhan dien va ghop ket qua.
    Ap dung co che Overlap (chong chong cheo) de chong mat chu o ranh gioi cat.
    """
    print(f"Dang tai file {input_wav_path}...")
    audio = AudioSegment.from_wav(input_wav_path)

    chunk_length_ms = chunk_minutes * 60 * 1000
    overlap_ms = 2 * 1000  # Chong cheo 2 giay
    total_length_ms = len(audio)
    total_chunks = math.ceil(total_length_ms / chunk_length_ms)

    print(f"Tong thoi luong: {total_length_ms / 1000 / 60:.2f} phut.")
    print(f"Se chia thanh {total_chunks} doan de xu ly.")

    full_transcript = []

    for i in range(total_chunks):
        start_ms = i * chunk_length_ms
        end_ms = min(total_length_ms, (i + 1) * chunk_length_ms + overlap_ms)

        print(f"  Doan {i + 1}/{total_chunks} ({start_ms/1000:.0f}s - {end_ms/1000:.0f}s)... ", end="", flush=True)

        chunk = audio[start_ms:end_ms]
        temp_chunk_path = f"temp_chunk_{i}.wav"
        chunk.export(temp_chunk_path, format="wav", parameters=["-acodec", "pcm_s16le"])

        try:
            with open(temp_chunk_path, "rb") as audio_file:
                transcription = client.audio.transcriptions.create(
                    model=model,
                    file=audio_file,
                    language="vi"
                )
            full_transcript.append(transcription.text)
            print("OK")
        except Exception as e:
            print(f"LOI: {e}")
        finally:
            if os.path.exists(temp_chunk_path):
                os.remove(temp_chunk_path)

        # Nghii 1 giay giua cac doan de tranh rate limit
        if i < total_chunks - 1:
            time.sleep(1)

    final_text = " ".join(full_transcript)
    print(f"\nHoan thanh! Tong so ky tu: {len(final_text)}")
    return final_text


audio_bai_giang = transcribe_long_audio(output_wav_path, chunk_minutes=10)
display(Markdown(f"**Response ({len(audio_bai_giang)} ky tu):** {audio_bai_giang[:500]}..."))

Dang tai file ../data/audio/test/CongThucLuongGiac.wav...
Tong thoi luong: 167.08 phut.
Se chia thanh 17 doan de xu ly.
  Doan 1/17 (0s - 602s)... OK
  Doan 2/17 (600s - 1202s)... OK
  Doan 3/17 (1200s - 1802s)... OK
  Doan 4/17 (1800s - 2402s)... OK
  Doan 5/17 (2400s - 3002s)... OK
  Doan 6/17 (3000s - 3602s)... OK
  Doan 7/17 (3600s - 4202s)... OK
  Doan 8/17 (4200s - 4802s)... OK
  Doan 9/17 (4800s - 5402s)... OK
  Doan 10/17 (5400s - 6002s)... OK
  Doan 11/17 (6000s - 6602s)... OK
  Doan 12/17 (6600s - 7202s)... OK
  Doan 13/17 (7200s - 7802s)... OK
  Doan 14/17 (7800s - 8402s)... OK
  Doan 15/17 (8400s - 9002s)... OK
  Doan 16/17 (9000s - 9602s)... OK
  Doan 17/17 (9600s - 10025s)... OK

Hoan thanh! Tong so ky tu: 108313


**Response (108313 ky tu):** thể record. nào hiện tại là chúng ta đang qua cái phần buổi thứ ba. của lớp nền tảng toán cho vật lý hiện tại là thầy đã dạy xong cho bạn về nội dung về phần vector phần nhân vô hướng và phần nhân có hướng rồi có đúng không và thêm nữa là thầy cũng đã gửi cho bạn các cái phần file bài tập về nhà để cho chúng ta làm thì những cái phần file bài tập về nhà thì thầy nhắc lại một lần nữa đó là những cái phần file đó nội dung nó không hề khó ở đây là mục tiêu của chúng ta, chúng ta cần phải làm những ...

In [3]:
# audio_bai_giang da duoc gan o cell tren

### Dùng LLM để tách thành các chủ đề con

In [7]:
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import os

load_dotenv()  # Load environment variables from .env file

api_key = os.getenv("OPENROUTER_API_KEY")

SYSTEM_PROMPT = """
Tôi sẽ cung cấp transcribe của một bài giảng sau
Bạn hãy tách ra thành các chủ đề con được nói tới trong bài giảng đó
Trả về kết quả dạng json:
{
    "topics": [
        {
            "topic": "Tên chủ đề con 1",
            "content": "Nội dung tóm tắt của chủ đề con 1"
        },
        {
            "topic": "Tên chủ đề con 2",
            "content": "Nội dung tóm tắt của chủ đề con 2"
        }
    ]
}

"""

# The client automatically picks up the OPENAI_API_KEY environment variable
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key)

response = client.chat.completions.create(
    model="deepseek/deepseek-v4-flash-0731:free", 
    messages=[
        # System prompt đặt ở đầu để định hướng phong cách trả lời cho AI
        {"role": "system", "content": SYSTEM_PROMPT},
        

        {"role": "user", "content": f"Transcribe của bài giảng: {audio_bai_giang}"}
    ],
    temperature=0.2,     
)


In [8]:
result = response.choices[0].message.content

In [9]:
print(result)

Dựa trên nội dung bài giảng, có thể tách thành các chủ đề chính như sau:

1. **Giới thiệu và mục đích bài học**  
   - Giáo viên nhấn mạnh tầm quan trọng của toán (lượ giác, hệ thức lượng) trong vật lý, yêu cầu học sinh học thuộc công thức và làm bài tập về nhà.

2. **Ứng dụng của hệ thức lượng và công thức lượng giác trong vật lý**  
   - Phân tích lực (ví dụ vật trượt trên mặt phẳng nghiêng).  
   - Dao động điều hòa (con lắc đơn, con lắc lò xo).  
   - Dòng điện xoay chiều (biểu thức cường độ dòng điện, hiệu điện thế).  
   - Giao thoa sóng (sóng mặt nước, sóng ánh sáng).

3. **Ôn tập hệ thức lượng trong tam giác vuông**  
   - Các công thức: \(c^2 = c' \cdot a\), \(b^2 = b' \cdot a\), \(a^2 = b^2 + c^2\), \(h^2 = b' \cdot c'\), \(\frac{1}{h^2} = \frac{1}{b^2} + \frac{1}{c^2}\).

4. **Định lý cosin**  
   - Công thức: \(a^2 = b^2 + c^2 - 2bc\cos\alpha\), \(b^2 = a^2 + c^2 - 2ac\cos\beta\), \(c^2 = a^2 + b^2 - 2ab\cos\gamma\).  
   - Chứng minh bằng cách sử dụng tích vô hướng của hai